# 第2章 Gymnasium の API の基本

第1章では、Stable-Baselines3 に CartPole を学習させ、棒を立て続けられるようにしました。そのとき、`model.predict(observation)` に「今の状態」を渡して行動を選ばせましたが、その状態が何なのかは説明しないままでした。

この章では、Gymnasium の環境を、学習アルゴリズムを通さずに自分の手で動かします。環境を作る、最初の状態に戻す、行動を1つ渡して1歩進める、という基本の操作を1つずつ確かめると、エージェントと環境の間でどんな値がやり取りされているかが見えてきます。最後に、強化学習を使わずに、自分で書いた1行の規則で棒を立ててみます。

- **前提**: [第1章](../ch01_first_training/ch01_first_training.ipynb)を終えていること
- **所要の目安**: 実行は CPU で数秒
- **使う装置**: CPU だけで動きます（ニューラルネットワークの学習はしません）

> **注記**
> - **出力の出どころ**: この Notebook に残っている出力は、著者の環境（Windows 11、CPU のみ、Gymnasium 1.3.0）で 2026-10-02 に実行したものです。各セルの出力がそのまま期待する結果で、直後の「期待する結果」の段落に読み方を書いています。
> - **進め方**: セルを上から順に実行します。

## 0. 学習目標と完了条件

この章を終えると、次のことができるようになります。

1. `gym.make()` で環境を作り、その環境がどんな部品（ラッパー）でできているかを確かめられる
2. 観測と行動の「空間」（`observation_space`・`action_space`）を読み、観測の4つの数の意味が分かる
3. `reset()` と `step()` の戻り値を読み、`terminated` と `truncated` の違いを説明できる
4. 自分で書いた規則（方策）で CartPole を動かし、成績を測れる

完了条件は、この Notebook を上から最後まで実行して、7節の規則で棒を 500 回立て続けられることです。

## 1. 全体像

強化学習では、エージェントと環境が、次のやり取りを繰り返します。

| 順番 | 向き | やり取りするもの | Gymnasium の操作 |
|---|---|---|---|
| 1 | 環境 → エージェント | 最初の観測 | `reset()` |
| 2 | エージェント → 環境 | 行動 | `step(action)` の引数 |
| 3 | 環境 → エージェント | 次の観測、報酬、終わったかどうか | `step(action)` の戻り値 |

2〜3 を、エピソードが終わるまで繰り返します。

ここで出てくる言葉は次のとおりです。

- **観測**（observation）: エージェントが見る、環境の今の様子を表す数。CartPole では4つの数（3節）
- **行動**（action）: エージェントが選んで環境に渡すもの。CartPole では「左へ押す」か「右へ押す」か
- **報酬**（reward）: 行動のあとに環境が返す、良し悪しを表す数。CartPole では、棒が立っている間、1歩ごとに 1

この章で使う Gymnasium のクラス（`Env`・`Wrapper`・`Space` など）の関係は、[クラス図](../../diagrams/gymnasium_classes.md)にまとめています。この章を読みながら、または読み終えてから見ると、部品どうしのつながりがつかみやすくなります。

## 2. 環境を作る

`gym.make()` に環境の名前（ID）を渡すと、環境ができます。できた環境を `print` すると、その作りが表示されます。

In [1]:
import gymnasium as gym
import numpy as np

env = gym.make("CartPole-v1")
print(env)
print("ID:", env.spec.id)
print("エピソードの上限の回数:", env.spec.max_episode_steps)
print("「解けた」とみなす報酬の目安:", env.spec.reward_threshold)

<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>
ID: CartPole-v1
エピソードの上限の回数: 500
「解けた」とみなす報酬の目安: 475.0


**期待する結果**（上の出力の読み方）: 1行目の `<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>` は、環境が入れ子になっていることを表しています。いちばん内側の `CartPoleEnv` が CartPole の本体で、その外側を3つの**ラッパー**（環境を包んで、働きを付け足す部品）が包んでいます。

| ラッパー | 付け足す働き |
|---|---|
| `PassiveEnvChecker` | `step()`・`reset()`・`render()` の呼び出しが Gymnasium の約束どおりかを確かめる |
| `OrderEnforcing` | `reset()` より先に `step()` や `render()` を呼ぶと、エラーにして順番の誤りを防ぐ |
| `TimeLimit` | エピソードが上限の回数（CartPole-v1 では 500）に達したら打ち切る |

`env.spec` は、`gym.make()` に登録されている環境の設定です。上限の回数は 500、「解けた」とみなす目安は報酬 475 です。

ラッパーは6節でもう一度扱います。

## 3. 観測と行動の「空間」を見る

**空間**（space）は、観測や行動として、どんな形の、どんな範囲の値がありうるかを表すものです。どの環境も、`observation_space`（観測の空間）と `action_space`（行動の空間）を持っています。

In [2]:
print("観測の空間:", env.observation_space)
print("観測の下限:", env.observation_space.low)
print("観測の上限:", env.observation_space.high)
print()
print("行動の空間:", env.action_space)
print("行動の数:", env.action_space.n)

観測の空間: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
観測の下限: [-4.8               -inf -0.41887903        -inf]
観測の上限: [4.8               inf 0.41887903        inf]

行動の空間: Discrete(2)
行動の数: 2


**期待する結果**（上の出力の読み方）: 観測の空間は `Box(..., (4,), float32)` です。**`Box`** は、実数を並べた配列の空間で、`(4,)` は4つの数が並ぶことを、`float32` はその数の型を表します。4つの数の意味と範囲は次のとおりです。

| 番号 | 意味 | 下限〜上限 |
|---|---|---|
| 0 | 台車の位置 | -4.8 〜 4.8 |
| 1 | 台車の速さ | 制限なし（`-inf` 〜 `inf`） |
| 2 | 棒の角度（ラジアン。正は右へ傾いた向き） | 約 -0.419 〜 0.419（約 ±24 度） |
| 3 | 棒の角速度（傾きが変わる速さ） | 制限なし |

エピソードが終わるのは、棒の角度が ±12 度（約 ±0.21 ラジアン）を超えたときか、台車の位置が ±2.4 を超えたときです。空間の範囲は、それより少し広めに取られています。

行動の空間は `Discrete(2)` です。**`Discrete`** は、0 から始まる整数のどれか1つを選ぶ空間で、`Discrete(2)` は 0 か 1 を表します。CartPole では、0 が台車を左へ押す、1 が右へ押す、という意味です。

In [3]:
env.action_space.seed(0)
print("でたらめに選んだ行動10個:", [int(env.action_space.sample()) for _ in range(10)])

でたらめに選んだ行動10個: [1, 1, 1, 0, 0, 0, 0, 0, 0, 1]


**期待する結果**（上の出力の読み方）: 0 と 1 がでたらめに並びます。`sample()` は、空間の中からでたらめに1つの値を選ぶ関数で、第0章のでたらめな動きはこれを使っていました。`seed(0)` で乱数の出方を固定しているので、同じ版のライブラリなら何度実行しても同じ並びになります。

## 4. `reset()` で最初の状態に戻す

`reset()` は、環境を最初の状態に戻し、最初の観測を返します。

In [4]:
observation, info = env.reset(seed=0)
print("最初の観測:", observation)
print("観測の空間に入っているか:", env.observation_space.contains(observation))
print("info:", info)

最初の観測: [ 0.01369617 -0.02302133 -0.04590265 -0.04834723]
観測の空間に入っているか: True
info: {}


**期待する結果**（上の出力の読み方）: 最初の観測は、0 に近い小さな4つの数です。CartPole は、台車も棒も、ほぼ真ん中でほぼ止まった状態から始まります（4つの数は、どれも ±0.05 の範囲から、でたらめに決まります）。`seed=0` を渡しているので、同じ版のライブラリなら、何度実行しても同じ値になります。

`contains()` は、値が空間の範囲に入っているかを確かめる関数です。`True` なので、この観測は3節の `Box` の範囲に入っています。

`reset()` は、観測と一緒に **`info`** という辞書も返します。環境によっては、学習には使わない補足の情報がここに入ります。CartPole では空（`{}`）です。

## 5. `step()` で1歩進める

`step()` に行動を渡すと、環境が1歩進み、5つの値が返ってきます。ここでは、右へ押す行動（1）を渡します。

In [5]:
observation, reward, terminated, truncated, info = env.step(1)
print("次の観測:", observation)
print("報酬:", reward)
print("terminated:", terminated)
print("truncated:", truncated)
print("info:", info)

次の観測: [ 0.01323574  0.17272775 -0.04686959 -0.3551522 ]
報酬: 1.0
terminated: False
truncated: False
info: {}


**期待する結果**（上の出力の読み方）: 5つの値の意味は次のとおりです。

| 値 | 意味 | この出力では |
|---|---|---|
| 観測 | 1歩進んだ後の状態 | 台車の速さ（番号1）が正、つまり右向きに変わった |
| 報酬 | この1歩で得た報酬 | 棒が立っているので 1.0 |
| `terminated` | 課題の決まりによってエピソードが終わったか（棒が倒れた等） | `False` |
| `truncated` | 課題の外の理由（回数の上限等）でエピソードが打ち切られたか | `False` |
| `info` | 補足の情報 | 空 |

右へ押したので台車の速さは右向きに変わり、その反動で、棒の角度（番号2）の負の値は少し大きくなっています（棒は左へ傾く向きに動いた）。棒の角速度（番号3）が負に大きくなっているのも、そのためです。

なお、台車の位置（番号0）は、0.0137 から 0.0132 へ、わずかに左へ動いています。CartPole は、位置を、押す前の速さ（`reset()` の直後は左向きの -0.023）で1歩分だけ進めてから、速さを更新するためです。右へ押した効果が位置に表れるのは、次の1歩からです。

### 5-1. `terminated` と `truncated` の違い

エピソードの終わり方は2種類あります。**`terminated`** は、棒が倒れた、ゴールに着いた、のように、課題そのものの決まりで終わった場合です。**`truncated`** は、課題の外の都合で打ち切った場合で、代表的なのは2節の `TimeLimit` による回数の上限です。

違いを見るために、上限の回数を 20 にした環境を作り、7節で紹介する「棒を倒さない規則」で動かします。

In [6]:
# 棒の角度と角速度を見て、傾きを打ち消す向きへ押す規則（7節で詳しく扱う）
def keep_balance(observation):
    angle, angular_velocity = observation[2], observation[3]
    return 1 if angle + 0.5 * angular_velocity > 0 else 0


short_env = gym.make("CartPole-v1", max_episode_steps=20)
observation, info = short_env.reset(seed=0)
steps = 0
while True:
    observation, reward, terminated, truncated, info = short_env.step(keep_balance(observation))
    steps += 1
    if terminated or truncated:
        break
print("続いた回数:", steps)
print("terminated:", terminated, " truncated:", truncated)

続いた回数: 20
terminated: False  truncated: True


**期待する結果**（上の出力の読み方）: 20 回で、`terminated=False`、`truncated=True` で終わります。棒は倒れていませんが、`max_episode_steps=20` の上限に達したので打ち切られました。

この2つを区別するのは、学習のときに意味が違うからです。棒が倒れて `terminated` になった場合、その先はもうありません。一方、`truncated` で打ち切った場合、本当はその先も続けられたはずです。学習アルゴリズムは、この違いを使って「この先にどれだけの報酬が見込めたか」の見積もりを変えます（Stable-Baselines3 は、この区別を内部で扱っています。著者が Stable-Baselines3 2.9.0 のソースで確かめました）。

## 6. ラッパーを見る

2節で見たラッパーの重なりを、外側から順にたどってみます。ラッパーは、包んでいる環境を `env` という属性に持っています。

In [7]:
layer = env
while hasattr(layer, "env"):
    print(type(layer).__name__, "→ 中の環境を包んでいる")
    layer = layer.env
print(type(layer).__name__, "← 本体")
print()
print("env.unwrapped:", type(env.unwrapped).__name__)

TimeLimit → 中の環境を包んでいる
OrderEnforcing → 中の環境を包んでいる
PassiveEnvChecker → 中の環境を包んでいる
CartPoleEnv ← 本体

env.unwrapped: CartPoleEnv


**期待する結果**（上の出力の読み方）: 外側から `TimeLimit` → `OrderEnforcing` → `PassiveEnvChecker` とたどり、最後に本体の `CartPoleEnv` に着きます。`env.unwrapped` は、すべてのラッパーを外した本体を直接取り出す属性です。

`hasattr(layer, "env")` は、`layer` が `env` という属性を持っているか（つまりラッパーか）を確かめる書き方です。

ラッパーは、自分で付け足すこともできます。Gymnasium には、よく使う働きのラッパーが用意されています。ここでは、エピソードごとの報酬の合計と長さを記録する `RecordEpisodeStatistics` を付けてみます。

In [8]:
from gymnasium.wrappers import RecordEpisodeStatistics

recorded = RecordEpisodeStatistics(gym.make("CartPole-v1"))
print(recorded)

observation, info = recorded.reset(seed=0)
recorded.action_space.seed(0)
while True:
    observation, reward, terminated, truncated, info = recorded.step(recorded.action_space.sample())
    if terminated or truncated:
        break
print("エピソードの記録:", {"報酬の合計": info["episode"]["r"], "長さ": info["episode"]["l"]})

<RecordEpisodeStatistics<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>>
エピソードの記録: {'報酬の合計': 18.0, '長さ': 18}


**期待する結果**（上の出力の読み方）: 1行目で、いちばん外側に `RecordEpisodeStatistics` が加わったことが分かります。でたらめな行動でエピソードを終えると、最後の `step()` の `info` に `"episode"` の項目が入り、報酬の合計（`r`）と長さ（`l`）が記録されています。CartPole では、報酬の合計と長さは同じ値になります。

第1章で使った Stable-Baselines3 の `Monitor` も、同じように環境を包んで、エピソードごとの報酬の合計と長さを記録するラッパーです。第1章の学習曲線は、この仕組みで記録した値を描いたものでした。

## 7. 遊んでみる：自分の規則で棒を立てる

ここまでの知識で、強化学習を使わずに、自分で書いた規則で CartPole を動かしてみます。観測の4つの数を見て行動を決める関数が、そのまま**方策**になります。

まず、いちばん素直な規則「棒が右へ傾いていたら右へ押し、左へ傾いていたら左へ押す」を試します。比べるために、5節で使った `keep_balance`（角度に、角速度を 0.5 倍して足したものの符号で押す向きを決める）も同じ条件で測ります。

In [9]:
# 棒の角度だけを見て、傾いている側へ押す規則
def push_toward_tilt(observation):
    return 1 if observation[2] > 0 else 0


# 方策（観測を受け取って行動を返す関数）で seed を変えて何回か動かし、続いた回数の一覧を返す
def run_policy(policy, seeds):
    lengths = []
    for seed in seeds:
        observation, info = env.reset(seed=seed)
        steps = 0
        while True:
            observation, reward, terminated, truncated, info = env.step(policy(observation))
            steps += 1
            if terminated or truncated:
                break
        lengths.append(steps)
    return lengths


for name, policy in [("push_toward_tilt", push_toward_tilt), ("keep_balance", keep_balance)]:
    lengths = run_policy(policy, seeds=range(100))
    print(f"{name:16s} 平均 {np.mean(lengths):6.1f}  最短 {min(lengths):3d}  最長 {max(lengths):3d}  500回に届いた数 {sum(n == 500 for n in lengths)}/100")

push_toward_tilt 平均   41.0  最短  25  最長  58  500回に届いた数 0/100


keep_balance     平均  500.0  最短 500  最長 500  500回に届いた数 100/100


**期待する結果**（上の出力の読み方）: seed を 0〜99 に変えて100回ずつ動かした結果です。

- `push_toward_tilt`（角度だけを見る）は、平均 41 回ほどで倒れます。でたらめ（第0章、平均 約 25 回）よりは長く続きますが、500 回には届きません
- `keep_balance`（角度と角速度を見る）は、100回とも 500 回に届きます

`run_policy` は、方策（観測を受け取って行動を返す関数）を受け取り、seed を変えながらエピソードを繰り返して、続いた回数を集める関数です。

角度だけを見る規則が倒れる様子を、著者が seed 0〜99 で調べたところ、100回とも、台車の位置ではなく棒の角度（±12 度）の条件で終わっていました。また、1回のエピソードの中で、棒の傾きの向きが左右に平均 3.4 回（2〜5回）入れ替わっていました。棒は左右に揺れながら、最後は角度の上限を超えて終わっている、ということです。

これは、傾いてから押し返すだけでは押し返しが遅れて、反対側へ行き過ぎてしまうためと考えられます。角速度も見る規則は、「今は右に傾いていても、左へ戻りつつある」ことが分かるので、その行き過ぎを防げると考えられます。

**やってみよう**: `keep_balance` の `0.5` を、`0.1` や `2.0` に変えて、もう一度上のセルを実行してみてください。結果は変わるでしょうか。台車の位置（番号0）や速さ（番号1）も使う規則を考えてみるのも面白いでしょう。

CartPole は、このように人が1行の規則を書いても解けるほど単純な課題です。では、なぜ強化学習を使うのでしょうか。後の章で扱う予定の環境（月面に着陸する LunarLander、二本足で歩く BipedalWalker など）では、観測の数も行動の選択肢も増え、うまくいく規則を人が書き下すのは難しくなっていきます。強化学習は、規則を人が書く代わりに、報酬を手がかりにして方策を自分で見つける方法です。第1章の PPO は、`keep_balance` のような規則を、誰にも教わらずにニューラルネットワークの形で見つけていた、と考えることができます。

## 8. まとめ

この章では、Gymnasium の環境の基本の操作を確かめました。

| 操作・属性 | 役割 | 節 |
|---|---|---|
| `gym.make(ID)` | 環境を作る（本体をラッパーで包んだものが返る） | 2節 |
| `env.spec` | 登録されている設定（上限の回数等） | 2節 |
| `observation_space`・`action_space` | 観測と行動の空間（`Box`・`Discrete`） | 3節 |
| `reset(seed=...)` | 最初の状態に戻し、最初の観測と `info` を返す | 4節 |
| `step(action)` | 1歩進め、観測・報酬・`terminated`・`truncated`・`info` を返す | 5節 |
| ラッパー・`unwrapped` | 環境に働きを付け足す部品と、本体の取り出し | 6節 |

第1章の `model.predict(observation)` に渡していたのは、3節で見た4つの数でした。これらのクラスの関係は、[Gymnasium のクラス図](../../diagrams/gymnasium_classes.md)で見られます。

- 次に読むもの: 概要資料（[強化学習](../../overview/rl.md)・[Gymnasium](../../overview/gymnasium.md)・[Stable-Baselines3](../../overview/sb3.md)・[PyTorch](../../overview/pytorch.md)）。ここまでの体験と、これからの章をつなぐ地図です
- その次: 第3章 FrozenLake（準備中）

<!-- TODO（著者用）: 第3章ができたらリンクにする -->

## 出典

この章は、次の公式の文書と、導入した Gymnasium 1.3.0 の動作（2026-10-02 確認）をもとに、著者の言葉でまとめたものです。逐語の転載ではありません。

- Farama Foundation, Gymnasium Documentation, Env: https://gymnasium.farama.org/api/env/
- Farama Foundation, Gymnasium Documentation, Spaces: https://gymnasium.farama.org/api/spaces/
- Farama Foundation, Gymnasium Documentation, Wrappers: https://gymnasium.farama.org/api/wrappers/
- Farama Foundation, Gymnasium Documentation（CartPole の環境の説明）: https://gymnasium.farama.org/environments/classic_control/cart_pole/
- Stable-Baselines3 2.9.0 のソース（`truncated` を `TimeLimit.truncated` として扱う処理。`stable_baselines3/common/vec_env/dummy_vec_env.py`・`stable_baselines3/common/on_policy_algorithm.py`）: https://github.com/DLR-RM/stable-baselines3/tree/v2.9.0/stable_baselines3/common

この教科書のライセンスと、使っている第三者のソフトウェアの一覧は、リポジトリの [LICENSE.md](../../LICENSE.md) にあります。